# EDA completo — train_2016_2017_union.csv (todas las tiendas)

`train_2016_2017_union.csv` es grande (~6 GB, decenas de millones de filas — todas las tiendas, 2016-2017), así que **no lo cargamos completo en memoria**. Lo recorremos **una sola vez, por partes (chunks)**, y en esa misma pasada:

- Calculamos **exacto** (sobre el 100% de las filas): nulos, medias/desviaciones, correlaciones, promedios de `unit_sales` por grupo, evolución mensual.
- Vamos guardando una **muestra acotada** (`SAMPLE_FRAC`) solo para los gráficos de distribución (histogramas, boxplot, dispersión) — es la única parte que usa muestreo, y es solo por legibilidad visual, no afecta ningún número reportado.

Va a tardar varios minutos en la celda del recorrido principal — imprime el avance.

## 0. Configuración inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
from collections import Counter, defaultdict

BASE_DIR = "C:/Tesis"
FILE_PATH = os.path.join(BASE_DIR, "train_2016_2017_union.csv")
CHUNK_SIZE = 2_000_000
SAMPLE_FRAC = 0.005     # ~0.5% de las filas -> apunta a unas 300.000 filas de muestra para graficar
RANDOM_STATE = 42

COLOR_PRINCIPAL = "#4C72B0"
COLOR_SECUNDARIO = "#DD8452"
MAPA_SECUENCIAL = "Blues"

plt.rcParams.update({
    "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.3, "font.size": 10,
})

tamano_gb = os.path.getsize(FILE_PATH) / (1024**3)
print(f"Tamaño en disco: {tamano_gb:.2f} GB")


## 1. Recorrido único por chunks — cálculos exactos + muestra para gráficos

Todo lo que vamos a necesitar se acumula acá en una sola pasada por el archivo, para no tener que volver a leerlo completo varias veces:

- Conteo de filas y de nulos por columna
- Chequeo de duplicados por `id` (vía monotonicidad — si el `id` viene siempre creciente, no hay duplicados, y es mucho más liviano que guardar todos los ids en memoria)
- Estadísticas de `unit_sales` (n, media, desviación, min, max, negativos)
- Sumas para correlación exacta `unit_sales` vs `dcoilwtico` y vs `transactions`
- Promedios de `unit_sales` por familia, por promoción, por feriado y por tienda
- Suma y conteo de `unit_sales` por mes (para la evolución temporal)
- Conteos de las variables categóricas
- Una muestra acotada y aleatoria (sin sesgo) para los gráficos de distribución

In [ ]:
DTYPES = {
    "id": "int64", "store_nbr": "int16", "item_nbr": "int32", "unit_sales": "float32",
    "cluster": "Int16", "class": "Int32", "perishable": "Int8", "transactions": "Int32",
}

# --- Acumuladores ---
filas_totales = 0
nulos_totales = None
id_monotonico = True
ultimo_id = -1

n_us = 0; suma_us = 0.0; sumasq_us = 0.0; min_us = np.inf; max_us = -np.inf; negativos_us = 0

# sumas para correlación: [n_pares, sum_x, sum_y, sum_xy, sum_x2, sum_y2]
corr_oil = [0, 0.0, 0.0, 0.0, 0.0, 0.0]
corr_trans = [0, 0.0, 0.0, 0.0, 0.0, 0.0]

grupo_family = defaultdict(lambda: [0.0, 0])       # {familia: [suma, cuenta]}
grupo_promo = defaultdict(lambda: [0.0, 0])
grupo_feriado = defaultdict(lambda: [0.0, 0])
grupo_tienda = defaultdict(lambda: [0.0, 0])
grupo_mes = defaultdict(lambda: [0.0, 0])

conteo_family = Counter()
conteo_store_type = Counter()
conteo_holiday_type = Counter()
conteo_onpromotion = Counter()
conteo_perishable = Counter()
conteo_cluster = Counter()
conteo_tiendas = Counter()

rng = np.random.default_rng(RANDOM_STATE)
partes_muestra = []


def acumular_grupo(acumulador, claves, valores):
    """Suma y cuenta unit_sales por clave, de forma vectorizada (mucho más rápido que un loop fila a fila)."""
    agregado = valores.groupby(claves).agg(["sum", "count"])
    for clave, (suma, cuenta) in agregado.iterrows():
        acumulador[clave][0] += suma
        acumulador[clave][1] += cuenta


for i, chunk in enumerate(pd.read_csv(FILE_PATH, dtype=DTYPES, parse_dates=["date"], chunksize=CHUNK_SIZE)):
    filas_totales += len(chunk)

    # Nulos
    nulos_chunk = chunk.isna().sum()
    nulos_totales = nulos_chunk if nulos_totales is None else nulos_totales.add(nulos_chunk, fill_value=0)

    # Duplicados por id (monotonicidad)
    ids = chunk["id"].to_numpy()
    if len(ids) > 0:
        if ids[0] <= ultimo_id or (len(ids) > 1 and not np.all(np.diff(ids) > 0)):
            id_monotonico = False
        ultimo_id = ids[-1]

    # unit_sales
    us = chunk["unit_sales"].dropna().to_numpy(dtype="float64")
    n_us += len(us)
    suma_us += us.sum()
    sumasq_us += (us ** 2).sum()
    if len(us) > 0:
        min_us = min(min_us, us.min())
        max_us = max(max_us, us.max())
    negativos_us += int((us < 0).sum())

    # Correlaciones (solo filas donde ambas variables no son nulas)
    for acumulador, col in [(corr_oil, "dcoilwtico"), (corr_trans, "transactions")]:
        pares = chunk[["unit_sales", col]].dropna()
        x = pares["unit_sales"].to_numpy(dtype="float64")
        y = pares[col].to_numpy(dtype="float64")
        acumulador[0] += len(x)
        acumulador[1] += x.sum()
        acumulador[2] += y.sum()
        acumulador[3] += (x * y).sum()
        acumulador[4] += (x ** 2).sum()
        acumulador[5] += (y ** 2).sum()

    # Promedios por grupo (vectorizado con groupby, no fila por fila)
    us_col = chunk["unit_sales"]
    acumular_grupo(grupo_family, chunk["family"], us_col)
    acumular_grupo(grupo_promo, chunk["onpromotion"].fillna("Sin dato"), us_col)
    acumular_grupo(grupo_feriado, chunk["holiday_type"].notna(), us_col)
    acumular_grupo(grupo_tienda, chunk["store_nbr"], us_col)
    acumular_grupo(grupo_mes, chunk["date"].dt.to_period("M"), us_col)

    # Conteos categóricos
    conteo_family.update(chunk["family"].dropna())
    conteo_store_type.update(chunk["store_type"].dropna())
    conteo_holiday_type.update(chunk["holiday_type"].dropna())
    conteo_onpromotion.update(chunk["onpromotion"].fillna("Sin dato"))
    conteo_perishable.update(chunk["perishable"].dropna())
    conteo_cluster.update(chunk["cluster"].dropna())
    conteo_tiendas.update(chunk["store_nbr"])

    # Muestra para gráficos
    mask = rng.random(len(chunk)) < SAMPLE_FRAC
    partes_muestra.append(chunk[mask])

    if (i + 1) % 5 == 0:
        print(f"Procesadas {filas_totales:,} filas...")

muestra = pd.concat(partes_muestra, ignore_index=True)
print(f"\nListo. Filas totales: {filas_totales:,} | Filas en la muestra para gráficos: {len(muestra):,}")


## 2. Vista general

In [ ]:
print(f"Filas totales: {filas_totales:,}")
print(f"¿id viene sin duplicados (monotónico)?: {'Sí' if id_monotonico else 'NO — revisar'}")
print("\nColumnas presentes:", list(muestra.columns))


## 3. Nulos por columna (exacto, sobre el 100% de las filas)
Es esperable ver nulos en `dcoilwtico` (fines de semana/feriados sin precio), en las columnas de `holidays_events` (la mayoría de los días no son feriado), y en `onpromotion`.

In [ ]:
resumen_nulos = pd.DataFrame({
    "nulos": nulos_totales.astype(int),
    "porcentaje": (nulos_totales / filas_totales * 100).round(2),
}).sort_values("nulos", ascending=False)
resumen_nulos


In [ ]:
con_nulos = resumen_nulos[resumen_nulos["nulos"] > 0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(con_nulos.index[::-1], con_nulos["porcentaje"][::-1], color=COLOR_PRINCIPAL)
ax.set_xlabel("% de filas nulas")
ax.set_title("Porcentaje de nulos por columna (exacto, 100% de los datos)")
plt.tight_layout()
plt.show()


## 4. Estadísticas exactas de unit_sales
Calculadas con las sumas acumuladas durante el recorrido — sobre el 100% de las filas, no la muestra.

In [ ]:
media_us = suma_us / n_us
var_us = (sumasq_us / n_us) - media_us ** 2
std_us = np.sqrt(max(var_us, 0))

print(f"n (no nulos): {n_us:,}")
print(f"Media: {media_us:.4f}")
print(f"Desviación estándar: {std_us:.4f}")
print(f"Mínimo: {min_us:.2f}")
print(f"Máximo: {max_us:.2f}")
print(f"Filas con unit_sales negativo (devoluciones): {negativos_us:,} ({negativos_us/n_us*100:.2f}%)")


## 5. Distribución de unit_sales (sobre la muestra, solo para el gráfico)
Los cuantiles/percentiles acá son aproximados (vienen de la muestra) — si más adelante necesitas la mediana o el rango intercuartil exactos sobre el 100% de los datos, se puede agregar un cálculo de dos pasadas (histograma acumulado).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(muestra["unit_sales"], bins=100, color=COLOR_PRINCIPAL)
axes[0].set_title("unit_sales (muestra, escala original)")
axes[0].set_xlabel("unit_sales")

axes[1].hist(np.log1p(muestra["unit_sales"].clip(lower=0)), bins=100, color=COLOR_SECUNDARIO)
axes[1].set_title("log(1 + unit_sales), muestra")
axes[1].set_xlabel("log1p(unit_sales)")

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(4, 5))
ax.boxplot(muestra["unit_sales"], vert=True)
ax.set_title("unit_sales — boxplot (muestra)")
plt.tight_layout()
plt.show()

q1, q3 = muestra["unit_sales"].quantile([0.25, 0.75])
print(f"Q1 (aprox., de la muestra): {q1:.2f} | Q3 (aprox.): {q3:.2f} | IQR (aprox.): {q3-q1:.2f}")


## 6. Evolución mensual de unit_sales (exacto)

In [ ]:
evolucion = pd.DataFrame([
    {"mes": str(mes), "suma": v[0], "promedio": v[0] / v[1], "n": v[1]}
    for mes, v in sorted(grupo_mes.items())
])
evolucion


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(evolucion["mes"], evolucion["suma"], marker="o", color=COLOR_PRINCIPAL)
ax.set_title("Suma de unit_sales por mes — todas las tiendas")
ax.set_xlabel("Mes")
ax.set_ylabel("Suma de unit_sales")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Variables categóricas (conteos exactos)

In [ ]:
top10_familias = pd.Series(conteo_family).sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(top10_familias.index[::-1], top10_familias.values[::-1], color=COLOR_PRINCIPAL)
ax.set_xlabel("Cantidad de filas")
ax.set_title("Top 10 familias de productos (conteo exacto)")
plt.tight_layout()
plt.show()


In [ ]:
print("Tiendas presentes:", len(conteo_tiendas), "de 54 en stores.csv")
print("store_type:", dict(conteo_store_type))
print("onpromotion:", dict(conteo_onpromotion))
print("perishable:", dict(conteo_perishable))
print("holiday_type (solo días feriado):", dict(conteo_holiday_type))


## 8. Promedios de unit_sales por grupo (exacto)

In [ ]:
promedio_por_familia = pd.Series({k: v[0]/v[1] for k, v in grupo_family.items()}).sort_values(ascending=False)
promedio_por_familia_top10 = promedio_por_familia.loc[top10_familias.index]

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(promedio_por_familia_top10.index[::-1], promedio_por_familia_top10.values[::-1], color=COLOR_PRINCIPAL)
ax.set_xlabel("unit_sales promedio")
ax.set_title("unit_sales promedio por familia (top 10 familias más frecuentes)")
plt.tight_layout()
plt.show()


In [ ]:
promedio_promocion = {k: v[0]/v[1] for k, v in grupo_promo.items()}
print("unit_sales promedio por onpromotion:", promedio_promocion)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar([str(k) for k in promedio_promocion.keys()], promedio_promocion.values(),
       color=[COLOR_PRINCIPAL, COLOR_SECUNDARIO, "#8C8C8C"][:len(promedio_promocion)])
ax.set_ylabel("unit_sales promedio")
ax.set_title("unit_sales promedio según onpromotion")
plt.tight_layout()
plt.show()


In [ ]:
promedio_feriado = {k: v[0]/v[1] for k, v in grupo_feriado.items()}
print("unit_sales promedio — feriado vs día normal:", promedio_feriado)

fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Día normal", "Feriado"], [promedio_feriado.get(False, 0), promedio_feriado.get(True, 0)],
       color=[COLOR_PRINCIPAL, COLOR_SECUNDARIO])
ax.set_ylabel("unit_sales promedio")
ax.set_title("unit_sales promedio: feriado vs. día normal")
plt.tight_layout()
plt.show()


In [ ]:
promedio_por_tienda = pd.Series({k: v[0]/v[1] for k, v in grupo_tienda.items()}).sort_values(ascending=False)
print("Top 10 tiendas por unit_sales promedio:")
promedio_por_tienda.head(10)


## 9. Correlaciones exactas de unit_sales
Calculadas con las sumas acumuladas de todo el archivo (fórmula de correlación de Pearson por sumas), no con la muestra.

In [ ]:
def correlacion_desde_sumas(acumulador):
    n, sx, sy, sxy, sx2, sy2 = acumulador
    num = n * sxy - sx * sy
    den = np.sqrt((n * sx2 - sx**2) * (n * sy2 - sy**2))
    return num / den if den != 0 else np.nan

correlacion_oil = correlacion_desde_sumas(corr_oil)
correlacion_trans = correlacion_desde_sumas(corr_trans)

print(f"Correlación unit_sales - dcoilwtico (exacta, n={corr_oil[0]:,}): {correlacion_oil:.4f}")
print(f"Correlación unit_sales - transactions (exacta, n={corr_trans[0]:,}): {correlacion_trans:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

muestra_oil = muestra.dropna(subset=["dcoilwtico"])
axes[0].scatter(muestra_oil["dcoilwtico"], muestra_oil["unit_sales"], alpha=0.1, s=6, color=COLOR_PRINCIPAL)
axes[0].set_xlabel("Precio del petróleo (dcoilwtico)")
axes[0].set_ylabel("unit_sales")
axes[0].set_title("unit_sales vs. petróleo (muestra)")

axes[1].scatter(muestra["transactions"], muestra["unit_sales"], alpha=0.1, s=6, color=COLOR_SECUNDARIO)
axes[1].set_xlabel("Transacciones de la tienda ese día")
axes[1].set_title("unit_sales vs. transacciones (muestra)")

plt.tight_layout()
plt.show()


## 10. Resumen ejecutivo
Todos los números clave juntos, calculados sobre el 100% de los datos — pensado para pegar directo en la sección de resultados de la tesis.

In [ ]:
print("===== RESUMEN EDA — train_2016_2017_union.csv (todas las tiendas) =====")
print(f"Filas totales: {filas_totales:,}")
print(f"¿id sin duplicados?: {'Sí' if id_monotonico else 'NO — revisar'}")
print(f"Tiendas presentes: {len(conteo_tiendas)} de 54")
print()
print(f"unit_sales -> media: {media_us:.2f} | std: {std_us:.2f} | min: {min_us:.2f} | max: {max_us:.2f}")
print(f"% de filas con unit_sales negativo (devoluciones): {negativos_us/n_us*100:.2f}%")
print()
print(f"unit_sales promedio EN promoción: {promedio_promocion.get(True, float('nan')):.2f}")
print(f"unit_sales promedio SIN promoción: {promedio_promocion.get(False, float('nan')):.2f}")
print(f"unit_sales promedio en FERIADO: {promedio_feriado.get(True, float('nan')):.2f}")
print(f"unit_sales promedio en día normal: {promedio_feriado.get(False, float('nan')):.2f}")
print()
print(f"Correlación unit_sales - dcoilwtico: {correlacion_oil:.4f}")
print(f"Correlación unit_sales - transactions: {correlacion_trans:.4f}")
print(f"Familia con mayor unit_sales promedio (top 10 más frecuentes): {promedio_por_familia_top10.index[0]}")
print(f"Tienda con mayor unit_sales promedio: {promedio_por_tienda.index[0]}")


---
## Notas para la tesis
- Todo lo de las secciones 2 a 4, 6, 7, 8 y 9 es **exacto** (100% de las filas, calculado en una sola pasada por chunks). Solo la sección 5 (histogramas/boxplot/cuantiles) y los gráficos de dispersión de la sección 9 usan la muestra acotada — es una limitación conocida y documentada, no un descuido.
- Los valores negativos de `unit_sales` son devoluciones, no errores.
- El marcado de feriado (`holiday_type` no nulo) es una simplificación — no distingue feriados locales de otra ciudad ni `transferred`.
- Si más adelante necesitas percentiles/mediana exactos (no aproximados desde la muestra), se puede agregar un cálculo de dos pasadas por el archivo.